# 最佳化器與學習率 (Optimizers & Learning Rate)

上一章中，我們已經學會如何使用 Loss Function 衡量模型的預測誤差，以及透過 `loss.backward()` 計算 Weight 與 Bias 的梯度。

但這裡還有一個問題：模型知道自己預測錯了，也知道梯度是多少，接下來應該如何調整 Weight 和 Bias，才能讓 Loss 逐漸下降？

這正是最佳化器 (Optimizer) 的工作。

## 從這裡開始

## 最佳化器 (Optimizer) 基本概念

最佳化器 (Optimizer) 是神經網路訓練時，根據梯度調整模型參數的演算法。

我們可以將 ANN 的訓練過程想像成學生準備考試：

| 神經網路概念          | 白話比喻       |
| --------------- | ---------- |
| Prediction      | 學生作答       |
| Target          | 正確答案       |
| Loss Function   | 計算答錯程度     |
| Backpropagation | 分析哪些地方造成錯誤 |
| Gradient        | 指出調整方向     |
| Optimizer       | 決定如何修改學習策略 |
| Learning Rate   | 每次調整的幅度    |

例如，模型要預測房屋租金：

```
真實租金：20,000 元
預測租金：25,000 元
```

Loss Function 會計算預測誤差。

Backpropagation 會計算每個 Weight 和 Bias 對 Loss 的影響。

Optimizer 則根據這些梯度更新參數，希望下一次預測更接近真實答案。

### 1.2 神經網路的完整訓練流程

```
輸入 Features
       ↓
Forward Pass
       ↓
Prediction
       ↓
Loss Function
       ↓
Loss
       ↓
Backward Pass
       ↓
計算 Gradient
       ↓
Optimizer
       ↓
更新 Weight / Bias
       ↓
下一次 Forward Pass
```

對應的 PyTorch 程式碼：

```
### 清除上一次的梯度optimizer.zero_grad()### 前向傳播predictions = model(X_train)### 計算 Lossloss = criterion(predictions, y_train)### 反向傳播，計算梯度loss.backward()### 使用 Optimizer 更新參數optimizer.step()
```

這五個步驟是後續 ANN、CNN、RNN 與 Transformer 訓練時都會用到的基本流程。

### 1.3 Optimizer 與 Loss Function 的差別

| 比較項目    | Loss Function         | Optimizer              |
| ------- | --------------------- | ---------------------- |
| 主要功能    | 衡量預測誤差                | 更新模型參數                 |
| 解決的問題   | 模型錯多少？                | 參數該怎麼調整？               |
| 常見選擇    | MSE、MAE、Cross Entropy | SGD、RMSprop、Adam、AdamW |
| PyTorch | `nn.MSELoss()`        | `optim.Adam()`         |
| 主要操作    | `criterion()`         | `optimizer.step()`     |

例如：

```
### 建立 Loss Functioncriterion = nn.MSELoss()### 建立 Optimizeroptimizer = torch.optim.Adam(    model.parameters(),    lr=0.001)
```

兩者通常需要一起使用，但它們負責不同的工作。

教材示意圖 01

- 圖片名稱：`01_Loss_Backpropagation_Optimizer.png`
- 圖片尺寸：1280 × 720 (16:9)
- 提示詞：製作大學深度學習教材流程圖，完整呈現 Input Features、ANN、Prediction、Loss Function、Loss、Backpropagation、Gradient、Optimizer、Update Weight and Bias，並用循環箭頭連接下一次訓練。藍色代表 Forward Pass，橘色代表 Backpropagation，綠色代表 Optimizer，附繁體中文與英文雙語標示，白色背景、專業扁平向量風格。

## 二、梯度下降法 (Gradient Descent)

### 2.1 什麼是梯度下降法？

梯度下降法 (Gradient Descent) 是深度學習中最重要的最佳化概念之一。

它的目標很簡單：

根據 Loss 的梯度，反覆調整模型參數，嘗試讓 Loss 逐漸降低。

可以想像自己站在一座山上，目標是走到山谷最低處。

但你無法一次看見整座山的地形，只能根據腳下的坡度，決定下一步往哪裡走。

在這個比喻中：

| 登山概念   | 深度學習          |
| ------ | ------------- |
| 山的高度   | Loss          |
| 所在位置   | Weight / Bias |
| 地面的坡度  | Gradient      |
| 下坡方向   | 負梯度方向         |
| 每一步的距離 | Learning Rate |
| 走路策略   | Optimizer     |

梯度下降法會沿著負梯度方向更新參數。

### 2.2 梯度下降法的公式

\\[ \boxed{ w\_{\text{new}}=w\_{\text{old}}-\eta\frac{\partial L}{\partial w} } \\]

其中：

| 符號                                  | 說明                |
| ----------------------------------- | ----------------- |
| \\(w\_{\text{old}}\\)               | 更新前的 Weight       |
| \\(w\_{\text{new}}\\)               | 更新後的 Weight       |
| \\(\eta\\)                          | Learning Rate     |
| \\(L\\)                             | Loss              |
| \\(\frac{\partial L}{\partial w}\\) | Loss 對 Weight 的梯度 |

這個公式不需要深入推導，重點是：

```
新的 Weight
=
舊的 Weight
-
Learning Rate × Gradient
```

### 2.3 梯度下降法的數值範例

假設：

```
Weight = 5
Gradient = 2
Learning Rate = 0.1
```

則：

\\[ w\_{\text{new}}=5-0.1\times2=4.8 \\]

Weight 從 5 更新成 4.8。

如果梯度是負數：

```
Weight = 5
Gradient = -2
Learning Rate = 0.1
```

則：

\\[ w\_{\text{new}}=5-0.1(-2)=5.2 \\]

Weight 會增加。

因此：

- Gradient 為正數時，普通梯度下降會降低該參數。
- Gradient 為負數時，普通梯度下降會提高該參數。
- Learning Rate 決定梯度對更新幅度的影響。

### 2.4 使用 Python 實作梯度下降法

先不使用 PyTorch Optimizer，而是自己寫出最基本的梯度下降法。

假設我們希望找到讓以下 Loss 最小的 Weight：

\\[ L(w)=(w-3)^2 \\]

這個函數的最低點在：

\\[ w=3 \\]

```
import matplotlib.pyplot as plt### 設定初始 Weightw = 8.0### 設定 Learning Ratelearning_rate = 0.1### 設定更新次數steps = 30### 儲存每次更新的 Weight 與 Lossweight_history = []loss_history = []### 開始梯度下降for step in range(steps):    ### 計算 Loss    loss = (w - 3) ** 2    ### 計算 Gradient    gradient = 2 * (w - 3)    ### 記錄目前狀態    weight_history.append(w)    loss_history.append(loss)    ### 更新 Weight    w = w - learning_rate * gradient### 顯示結果print("最後的 Weight：", w)print("最後的 Loss：", (w - 3) ** 2)### 視覺化 Loss 變化plt.figure(figsize=(9, 5))plt.plot(    loss_history,    marker="o",    label="Loss")plt.title("Gradient Descent")plt.xlabel("Step")plt.ylabel("Loss")plt.grid(alpha=0.3)plt.legend()plt.show()
```

執行後，Weight 會逐漸接近 3，而 Loss 會逐漸接近 0。

這就是梯度下降法最基本的運作方式。

### 2.5 梯度下降不保證找到全域最佳解

在上面的例子中，Loss 是一條簡單的 U 型曲線。

但真正的深度神經網路可能具有非常複雜的損失地形 (Loss Landscape)，包含：

- 多個局部極小值。
- 鞍點 (Saddle Points)。
- 平坦區域。
- 梯度很大的區域。

因此，梯度下降法不保證每次都能找到全域最低的 Loss。

實際訓練還會受到模型初始化、資料、Learning Rate、Optimizer 與 Batch Size 等因素影響。

教材示意圖 02

- 圖片名稱：`02_Gradient_Descent.png`
- 圖片尺寸：1280 × 720 (16:9)
- 提示詞：繪製 Gradient Descent 教學圖，使用 U 型 Loss 曲線，X 軸為 Weight，Y 軸為 Loss，顯示一顆小球沿著曲線逐步下降到最低點，使用箭頭標示 Gradient、Negative Gradient Direction、Learning Rate、Minimum Loss，附繁體中文解釋，白底、16:9。

## 三、三種梯度下降方式

梯度下降法可以依照「每次計算梯度時使用多少筆訓練資料」分成三種常見方式。

分別是：

1. Batch Gradient Descent (批次梯度下降)
2. Stochastic Gradient Descent (隨機梯度下降)
3. Mini-batch Gradient Descent (小批次梯度下降)

這裡需要先釐清一個觀念：

Batch、Stochastic、Mini-batch 描述的是每次更新使用多少筆資料；SGD、Adam、AdamW 則描述參數如何根據梯度更新。

這兩個概念可以搭配使用，並不是互斥的選項。

### 3.1 Batch Gradient Descent (批次梯度下降)

Batch Gradient Descent 每次使用整個訓練資料集計算梯度，再更新一次模型參數。

例如：

```
Train Set：1,000 筆

第 1 次更新：
使用全部 1,000 筆
       ↓
計算平均 Loss
       ↓
Backpropagation
       ↓
更新 Weight
```

如果訓練 10 個 Epoch，每個 Epoch 更新一次：

```
總更新次數 = 10
```

優點

- 每次梯度根據完整訓練資料計算。
- 對固定模型參數而言，梯度較穩定。
- 適合資料量不大的情況。

缺點

- 資料量大時，記憶體需求可能很高。
- 每次參數更新需要處理所有資料。
- 更新頻率較低。

### 3.2 Stochastic Gradient Descent (隨機梯度下降)

Stochastic Gradient Descent 在嚴格的單樣本形式中，每次只使用一筆資料計算梯度並更新參數。

例如：

```
Train Set：1,000 筆

第 1 筆 → 計算梯度 → 更新
第 2 筆 → 計算梯度 → 更新
第 3 筆 → 計算梯度 → 更新
...
第 1000 筆 → 計算梯度 → 更新
```

因此，一個 Epoch 可能有 1,000 次更新。

優點

- 每次更新只需要處理少量資料。
- 更新頻率高。
- 梯度的隨機波動有時有助於探索損失地形。

缺點

- 梯度波動較大。
- Loss 可能上下震盪。
- 單筆運算通常無法充分利用 GPU 的平行處理能力。

### 3.3 Mini-batch Gradient Descent (小批次梯度下降)

Mini-batch Gradient Descent 每次使用一小部分資料計算梯度。

例如：

```
Train Set：1,000 筆
Batch Size：100
```

每次處理 100 筆資料：

```
Batch 1：第 1～100 筆 → 更新
Batch 2：第 101～200 筆 → 更新
Batch 3：第 201～300 筆 → 更新
...
Batch 10：第 901～1000 筆 → 更新
```

一個 Epoch 有 10 次更新。

Mini-batch 是現代深度學習最常見的訓練方式。

因為它在記憶體使用、計算效率與梯度穩定性之間取得實用的平衡。

### 3.4 三種方式比較

| 比較項目     | Batch    | Stochastic | Mini-batch |
| -------- | -------- | ---------- | ---------- |
| 每次使用資料量  | 全部訓練資料   | 1 筆        | 一小批        |
| 梯度波動     | 較小       | 較大         | 介於兩者之間     |
| 更新頻率     | 低        | 高          | 中等         |
| GPU 平行運算 | 可能受記憶體限制 | 通常效率較低     | 通常較有效率     |
| 適用情境     | 小型資料集    | 特殊線上學習或教學  | 多數深度學習訓練   |

### 3.5 Batch Size 與 Iteration 的關係

假設：

```
Train Set：1,000 筆
Batch Size：128
```

那麼：

\\[ \text{Iterations per Epoch} = \left\lceil\frac{1000}{128}\right\rceil =8 \\]

其中最後一個 Batch 只有 104 筆。

可以使用 Python 計算：

```
import math### 設定資料筆數n_samples = 1000### 設定 Batch Sizebatch_size = 128### 計算每個 Epoch 的 Iterationsiterations = math.ceil(    n_samples / batch_size)### 顯示結果print("Iterations per Epoch：", iterations)
```

如果 `DataLoader(drop_last=True)`，最後不足一個 Batch 的資料會被捨棄，此時計算方式就會不同。

### 3.6 使用 DataLoader 實作三種訓練方式

```
import torchfrom torch.utils.data import TensorDataset, DataLoader### 建立模擬資料X = torch.randn(1000, 5)y = torch.randn(1000, 1)### 建立 TensorDatasetdataset = TensorDataset(X, y)### Batch Gradient Descentbatch_loader = DataLoader(    dataset,    batch_size=len(dataset),    shuffle=True)### Stochastic Gradient Descentstochastic_loader = DataLoader(    dataset,    batch_size=1,    shuffle=True)### Mini-batch Gradient Descentmini_batch_loader = DataLoader(    dataset,    batch_size=64,    shuffle=True)### 比較每個 Epoch 的 Batch 數量print("Batch：", len(batch_loader))print("Stochastic：", len(stochastic_loader))print("Mini-batch：", len(mini_batch_loader))
```

這裡的 `DataLoader` 只是決定資料如何分批。

真正使用哪一種參數更新演算法，還要看後續選擇的 Optimizer。

### 3.7 DataLoader 常用參數

```
DataLoader(    dataset,    batch_size=64,    shuffle=True,    num_workers=0,    drop_last=False)
```

| 參數            | 說明                      |
| ------------- | ----------------------- |
| `dataset`     | 要載入的資料集                 |
| `batch_size`  | 每個 Batch 包含多少筆資料        |
| `shuffle`     | 是否在每次建立迭代器時重新打亂資料順序     |
| `num_workers` | 資料載入使用的子程序數量            |
| `drop_last`   | 是否捨棄最後不足 Batch Size 的資料 |

Windows + Jupyter Notebook 環境下，初學者可以先使用 `num_workers=0`，減少多程序載入的相容性問題。

教材示意圖 03

- 圖片名稱：`03_Batch_Stochastic_MiniBatch.png`
- 圖片尺寸：1280 × 720 (16:9)
- 提示詞：製作三欄對照圖，展示 Batch Gradient Descent、Stochastic Gradient Descent、Mini-batch Gradient Descent。使用 12 個小資料方塊表示資料集，Batch 一次使用全部 12 筆，Stochastic 每次使用 1 筆，Mini-batch 每次使用 4 筆，並以箭頭標示每次更新 Weight 的時機。繁體中文與英文雙語標籤，白底、藍橘綠三色，16:9。

## 四、SGD (Stochastic Gradient Descent)

### 4.1 什麼是 SGD？

SGD 是 PyTorch 中最基本的最佳化器之一。

它會根據目前的梯度與 Learning Rate 更新模型參數。

基本更新公式：

\\[ w\_{\text{new}}=w\_{\text{old}}-\eta g \\]

其中 \\(g\\) 代表目前的梯度。

在 PyTorch 中：

```
optimizer = torch.optim.SGD(    model.parameters(),    lr=0.01)
```

就能建立 SGD Optimizer。

這裡有一個容易混淆的地方：

`torch.optim.SGD` 雖然名稱是 Stochastic Gradient Descent，但它也能使用 Full-Batch 或 Mini-Batch 梯度。

SGD 是更新演算法；Batch Size 是每次用多少資料計算梯度。

### 4.2 SGD 的運作方式

假設目前：

```
Weight = 5
Gradient = 2
Learning Rate = 0.1
```

則：

```
Weight = 5 - 0.1 × 2
       = 4.8
```

SGD 根據目前梯度更新 Weight。

這種基本形式沒有使用過去梯度的累積方向。

因此，在某些狹長或彎曲的 Loss Landscape 中，可能出現來回震盪的現象。

### 4.3 使用 PyTorch 建立 SGD

```
import torch.optim as optim### 建立 SGD Optimizeroptimizer = optim.SGD(    model.parameters(), # 指定要更新的模型參數    lr=0.01, # 學習率    momentum=0.0, # 不使用 Momentum    weight_decay=0.0 # 不使用 Weight Decay)
```

### 4.4 SGD 主要參數

| 參數               | 預設值     | 說明                     |
| ---------------- | ------- | ---------------------- |
| `params`         | 必填      | 需要更新的模型參數              |
| `lr`             | 必填      | Learning Rate          |
| `momentum`       | `0`     | 動量係數                   |
| `dampening`      | `0`     | 動量更新時的阻尼係數             |
| `weight_decay`   | `0`     | L2 型權重衰減               |
| `nesterov`       | `False` | 是否使用 Nesterov Momentum |
| `maximize`       | `False` | 是否改成最大化目標              |
| `foreach`        | `None`  | 是否使用批次化 Tensor 更新實作    |
| `differentiable` | `False` | 是否追蹤 Optimizer 更新步驟的梯度 |
| `fused`          | `None`  | 是否使用支援的融合式更新實作         |

初學階段先掌握 `lr`、`momentum` 與 `weight_decay` 即可。

### 4.5 SGD 的優缺點

| 優點           | 缺點                          |
| ------------ | --------------------------- |
| 原理簡單         | 對 Learning Rate 較敏感         |
| 容易理解參數更新     | 在某些損失地形中可能震盪                |
| 記憶體額外負擔較小    | 有時需要較多更新次數                  |
| 適合建立最佳化器基礎觀念 | 可能需要搭配 Momentum 與 Scheduler |

## 五、Momentum (動量)

### 5.1 什麼是 Momentum？

Momentum 是 SGD 常見的改良方法。

一般 SGD 主要根據目前梯度決定更新方向。

Momentum 則會考慮過去的更新趨勢，讓參數更新具有某種慣性。

可以想像一顆球從山坡往下滾。

普通 SGD：

```
每走一步
    ↓
根據目前坡度
    ↓
決定下一步方向
```

Momentum：

```
每走一步
    ↓
考慮目前坡度
    +
過去累積的移動趨勢
    ↓
決定下一步方向
```

這有助於減少部分方向上的來回震盪，並在方向一致時加快移動。

### 5.2 Momentum 的直覺範例

假設模型的梯度方向一直往右：

```
Step 1：→
Step 2：→
Step 3：→
Step 4：→
```

Momentum 會累積這個方向的更新趨勢。

但如果梯度在某個方向反覆改變：

```
Step 1：↑
Step 2：↓
Step 3：↑
Step 4：↓
```

動量累積可能抵銷部分震盪。

這也是 Momentum 在某些狹長谷地中能改善 SGD 訓練效率的原因。

### 5.3 使用 PyTorch 建立 Momentum SGD

```
import torch.optim as optim### 建立 SGD + Momentumoptimizer = optim.SGD(    model.parameters(),    lr=0.01, # 學習率    momentum=0.9 # 動量係數)
```

這仍然是 `SGD` Optimizer，只是開啟 Momentum。

### 5.4 Momentum 參數說明

```
momentum=0.9
```

代表過去梯度累積的影響會以係數 0.9 延續。

常見設定：

| Momentum | 說明              |
| -------- | --------------- |
| `0.0`    | 不使用 Momentum    |
| `0.5`    | 較弱的動量           |
| `0.9`    | 常見起始設定          |
| `0.99`   | 較強的動量，可能需要更謹慎調整 |

Momentum 越大，不代表訓練一定越快或越準確。

如果累積的更新方向不合適，也可能造成過衝 (Overshooting)。

### 5.5 Nesterov Momentum

PyTorch 也支援 Nesterov Momentum。

```
optimizer = optim.SGD(    model.parameters(),    lr=0.01,    momentum=0.9,    nesterov=True)
```

它使用與一般 Momentum 不同的梯度更新方式，通常可以理解為帶有前瞻修正的動量方法。

本章先知道它是 SGD 的延伸選項即可，不需要深入推導。

教材示意圖 04

- 圖片名稱：`04_SGD_vs_Momentum.png`
- 圖片尺寸：1280 × 720 (16:9)
- 提示詞：繪製 SGD 與 Momentum 在狹長 U 型谷地中的最佳化路徑比較圖。左側 SGD 使用多次左右震盪的折線逐漸前進，右側 Momentum 使用較平順的路徑朝最低點移動。清楚標示 Start、Minimum Loss、SGD、Momentum，使用藍色與橘色箭頭，繁體中文教學註解，白底、16:9。

## 六、RMSprop (Root Mean Square Propagation)

### 6.1 什麼是 RMSprop？

SGD 與 Momentum 都會受到 Learning Rate 的影響。

但神經網路可能包含大量不同的 Weight，每個參數的梯度大小也不一樣。

例如：

```
Weight A 的 Gradient = 0.001
Weight B 的 Gradient = 100
```

如果所有參數都使用相同的基本步長比例，可能出現：

- 某些參數更新幅度很小。
- 某些參數更新幅度很大。
- 訓練過程不容易穩定。

RMSprop 的主要想法是：

根據各個參數最近一段時間的平方梯度大小，自動調整它們的有效更新尺度。

梯度平方的移動平均較大時，對應的更新會受到較強的縮放。

梯度平方的移動平均較小時，對應的更新受到的縮放較弱。

因此 RMSprop 屬於自適應學習率 (Adaptive Learning Rate) 最佳化器。

### 6.2 RMSprop 如何運作？

可以簡化成：

```
取得目前 Gradient
        ↓
計算 Gradient²
        ↓
更新平方梯度的移動平均
        ↓
根據移動平均調整更新尺度
        ↓
更新 Weight / Bias
```

它並不是直接為每個參數設定一個固定的新 Learning Rate，而是透過梯度統計量縮放更新。

### 6.3 使用 PyTorch 建立 RMSprop

```
import torch.optim as optim### 建立 RMSprop Optimizeroptimizer = optim.RMSprop(    model.parameters(),    lr=0.001, # 基礎學習率    alpha=0.99, # 平方梯度移動平均係數    eps=1e-8, # 避免分母接近 0    weight_decay=0, # 權重衰減    momentum=0.0, # 是否搭配動量    centered=False # 是否使用中心化 RMSprop)
```

### 6.4 RMSprop 參數說明

| 參數             | 預設值     | 說明            |
| -------------- | ------- | ------------- |
| `lr`           | `0.01`  | 基礎學習率         |
| `alpha`        | `0.99`  | 平方梯度移動平均係數    |
| `eps`          | `1e-8`  | 數值穩定項         |
| `weight_decay` | `0`     | L2 型權重衰減      |
| `momentum`     | `0`     | 是否搭配動量        |
| `centered`     | `False` | 是否使用中心化梯度變異估計 |

注意：PyTorch RMSprop 的預設 `lr` 是 `0.01`。本章實驗會明確指定 `0.001`，避免不同 Optimizer 的預設值造成不必要的差異。

### 6.5 alpha 代表什麼？

`alpha` 控制過去平方梯度統計量的保留程度。

例如：

```
alpha=0.99
```

代表過去累積的平方梯度資訊保留較多，統計量通常變化得比較平緩。

如果改成：

```
alpha=0.9
```

就會讓最近的梯度平方對統計量產生較大的影響。

但 `alpha` 並不是越高越好，需要配合資料與訓練狀況。

### 6.6 RMSprop 的優缺點

| 優點              | 缺點                |
| --------------- | ----------------- |
| 能自適應縮放不同參數的更新   | 需要維護額外的梯度統計量      |
| 適合梯度尺度差異較大的情境   | 對超參數仍然敏感          |
| 有時比基本 SGD 更容易訓練 | 不保證比 Adam 或 SGD 好 |
| 可搭配 Momentum    | 需要額外記憶體           |

## 七、Adam (Adaptive Moment Estimation)

### 7.1 什麼是 Adam？

Adam 是深度學習中非常常見的 Optimizer。

它結合兩種重要概念：

- 類似 Momentum 的一階梯度移動平均。
- 類似 RMSprop 的二階平方梯度移動平均。

因此 Adam 不只考慮梯度方向，也會根據梯度大小的歷史統計調整更新尺度。

白話來說：

Adam 會同時參考「最近往哪個方向走」與「不同方向的梯度通常有多大」。

### 7.2 Adam 的運作流程

```
目前 Gradient
      ↓
計算一階動量
(梯度的移動平均)
      ↓
計算二階動量
(平方梯度的移動平均)
      ↓
進行偏差修正
      ↓
計算參數更新量
      ↓
更新 Weight / Bias
```

一階動量可以理解成梯度方向的累積趨勢。

二階動量則用來衡量梯度平方大小的歷史統計。

Adam 會利用這兩種資訊決定更新方式。

### 7.3 使用 PyTorch 建立 Adam

```
import torch.optim as optim### 建立 Adam Optimizeroptimizer = optim.Adam(    model.parameters(),    lr=0.001, # 學習率    betas=(0.9, 0.999), # 一階與二階動量係數    eps=1e-8, # 數值穩定項    weight_decay=0.0 # 不使用權重衰減)
```

### 7.4 Adam 參數說明

| 參數             | 預設值            | 說明              |
| -------------- | -------------- | --------------- |
| `lr`           | `0.001`        | 基礎學習率           |
| `betas`        | `(0.9, 0.999)` | 一階、二階移動平均係數     |
| `eps`          | `1e-8`         | 避免數值不穩定         |
| `weight_decay` | `0`            | 權重衰減係數          |
| `amsgrad`      | `False`        | 是否使用 AMSGrad 變體 |
| `maximize`     | `False`        | 是否最大化目標         |
| `foreach`      | `None`         | 批次化更新實作選項       |
| `fused`        | `None`         | 融合式更新實作選項       |

### 7.5 betas 參數如何理解？

```
betas=(0.9, 0.999)
```

這是 Adam 常見的預設設定。

其中：

| 參數            | 主要作用       |
| ------------- | ---------- |
| `beta1=0.9`   | 控制一階梯度移動平均 |
| `beta2=0.999` | 控制平方梯度移動平均 |

`beta1` 越接近 1，過去的梯度方向資訊通常保留得越久。

`beta2` 越接近 1，平方梯度統計量通常變化得越平緩。

初學階段通常不需要先調整 `betas`，可以從預設值開始。

### 7.6 Adam 的優缺點

| 優點                          | 缺點                      |
| --------------------------- | ----------------------- |
| 常能快速得到可用的訓練結果               | 需要額外記憶體                 |
| 對不同參數自適應縮放更新                | 不保證最佳泛化表現               |
| 適合許多 ANN、CNN、Transformer 任務 | 仍需調整 Learning Rate      |
| 常作為實驗起始選擇                   | 某些任務可能不如 SGD + Momentum |

Adam 是實用的起點，但不是所有模型的最佳答案。

## 八、AdamW (Adam with Decoupled Weight Decay)

### 8.1 什麼是 AdamW？

AdamW 是 Adam 的常見延伸版本。

它主要改善 Adam 中 Weight Decay 的處理方式。

Weight Decay (權重衰減) 可以理解成：訓練時對模型權重施加額外的縮減效果，避免某些參數變得過大。

在一般 Adam 中，傳統 L2 型權重衰減會與梯度結合，受到 Adam 自適應縮放機制影響。

AdamW 則將 Weight Decay 與梯度更新解耦 (Decoupled)。

因此，Weight Decay 不會以相同方式混入 Adam 的梯度動量統計。

### 8.2 Adam 與 AdamW 的白話比較

Adam：

```
Gradient
   +
L2 型 Weight Decay
       ↓
Adam 自適應更新
       ↓
更新參數
```

AdamW：

```
Gradient
       ↓
Adam 自適應更新
       +
獨立的 Weight Decay
       ↓
更新參數
```

這樣可以讓權重衰減的效果更容易與 Adam 的梯度縮放機制分開控制。

### 8.3 使用 PyTorch 建立 AdamW

```
import torch.optim as optim### 建立 AdamW Optimizeroptimizer = optim.AdamW(    model.parameters(),    lr=0.001, # 學習率    betas=(0.9, 0.999), # 動量係數    eps=1e-8, # 數值穩定項    weight_decay=0.01 # 權重衰減係數)
```

### 8.4 AdamW 參數說明

| 參數             | 預設值            | 說明           |
| -------------- | -------------- | ------------ |
| `lr`           | `0.001`        | 學習率          |
| `betas`        | `(0.9, 0.999)` | 一階與二階動量係數    |
| `eps`          | `1e-8`         | 數值穩定項        |
| `weight_decay` | `0.01`         | 解耦式權重衰減係數    |
| `amsgrad`      | `False`        | 是否使用 AMSGrad |

注意，AdamW 的預設 `weight_decay=0.01`，而 Adam 的預設值為 `0`。

因此，如果直接使用兩者預設參數比較，實際上不只改變了 Optimizer，也改變了正則化設定。

這在實驗設計時必須特別注意。

### 8.5 AdamW 一定比 Adam 好嗎？

不一定。

AdamW 的解耦式 Weight Decay 在許多深度學習任務中很實用，但它不保證每個資料集都比 Adam 更準確。

例如：

- 資料量很少時，正則化強度可能影響模型學習。
- 模型本身很簡單時，Adam 與 AdamW 可能差異不大。
- Weight Decay 設定過強時，可能導致欠擬合。

因此，仍然需要透過 Validation Set 比較。

教材示意圖 05

- 圖片名稱：`05_SGD_RMSprop_Adam_AdamW.png`
- 圖片尺寸：1280 × 720 (16:9)
- 提示詞：製作四種最佳化器原理比較資訊圖，分成 SGD、RMSprop、Adam、AdamW 四個區塊。SGD 標示 Current Gradient，RMSprop 標示 Squared Gradient Moving Average，Adam 標示 First and Second Moments，AdamW 標示 Adam + Decoupled Weight Decay。每個區塊有簡潔流程箭頭、繁體中文白話解釋，白底、16:9。

## 九、最佳化器比較 (Optimizer Comparison)

### 9.1 五種 Optimizer 整理

| Optimizer      | 核心概念           | 常見起始 Learning Rate | 特點          |
| -------------- | -------------- | ------------------ | ----------- |
| SGD            | 使用目前梯度更新       | 0.01               | 簡單、額外記憶體需求低 |
| SGD + Momentum | 累積更新趨勢         | 0.01               | 可減少部分方向的震盪  |
| RMSprop        | 根據平方梯度統計縮放更新   | 0.001              | 自適應更新       |
| Adam           | 一階與二階動量        | 0.001              | 常見的實用起點     |
| AdamW          | Adam + 解耦式權重衰減 | 0.001              | 方便搭配權重衰減    |

表中的 Learning Rate 只是常見實驗起點，不是固定最佳值。

### 9.2 什麼情況適合哪個 Optimizer？

| 情境                   | 可以先嘗試                    |
| -------------------- | ------------------------ |
| 初學 PyTorch           | SGD                      |
| 想觀察 Momentum 的效果     | SGD + Momentum           |
| 梯度尺度差異較大             | RMSprop                  |
| 一般 ANN 快速建立基準模型      | Adam                     |
| 希望搭配解耦式 Weight Decay | AdamW                    |
| 訓練大型神經網路             | AdamW 或其他適合架構的 Optimizer |

實務上沒有必要把所有 Optimizer 都當成必須搭配的工具。

通常是選擇一個 Optimizer 訓練模型，再根據驗證結果決定是否更換。

## 十、學習率 (Learning Rate)

### 10.1 什麼是 Learning Rate？

Learning Rate 是 Optimizer 中最重要的超參數之一。

它決定梯度對參數更新幅度的影響程度。

在基本 SGD 中：

\\[ w\_{\text{new}}=w\_{\text{old}}-\eta g \\]

其中 \\(\eta\\) 就是 Learning Rate。

可以把它想像成下山時每一步跨出去的距離。

### 10.2 Learning Rate 太小

假設：

```
learning_rate = 0.000001
```

每次參數更新可能非常小。

可能出現：

- Loss 下降緩慢。
- 訓練需要更多更新次數。
- 在有限訓練時間內表現不佳。

但 Learning Rate 小不一定是壞事，在訓練後期或微調模型時，小 Learning Rate 可能很有幫助。

### 10.3 Learning Rate 太大

假設：

```
learning_rate = 1.0
```

每次更新可能跨越很大的距離。

可能出現：

- Loss 大幅震盪。
- 參數更新越過較好的區域。
- Loss 持續上升。
- 嚴重時出現 NaN 或 Infinity。

### 10.4 合適的 Learning Rate

合適的 Learning Rate 通常能讓模型在合理的訓練時間內穩定降低 Loss。

但不存在所有任務通用的最佳值。

| Learning Rate | 可能觀察到的現象    |
| ------------- | ----------- |
| 太小            | Loss 下降很慢   |
| 適中            | Loss 相對穩定下降 |
| 太大            | Loss 震盪甚至發散 |

### 10.5 使用 Python 比較 Learning Rate

下面延續前面的簡單函數：

\\[ L(w)=(w-3)^2 \\]

```
import matplotlib.pyplot as plt### 建立梯度下降實驗函式def gradient_descent_experiment(learning_rate, steps=25):    ### 初始 Weight    w = 8.0    ### 儲存 Loss    losses = []    ### 開始更新    for step in range(steps):        ### 計算 Loss        loss = (w - 3) ** 2        ### 計算 Gradient        gradient = 2 * (w - 3)        ### 儲存 Loss        losses.append(loss)        ### 更新 Weight        w = w - learning_rate * gradient    return losses### 比較三種 Learning Ratelearning_rates = [    0.01,    0.1,    1.1]### 建立畫布plt.figure(figsize=(10, 6))### 繪製 Loss 曲線for lr in learning_rates:    losses = gradient_descent_experiment(        learning_rate=lr    )    plt.plot(        losses,        label=f"LR = {lr}"    )### 設定圖表plt.title("Learning Rate Comparison")plt.xlabel("Step")plt.ylabel("Loss")plt.yscale("log") # 使用對數尺度觀察不同量級plt.legend()plt.grid(alpha=0.3)### 顯示圖表plt.show()
```

這個例子中：

- `0.01` 下降較慢。
- `0.1` 能穩定下降。
- `1.1` 會出現逐漸放大的震盪。

但這只是特定二次函數的結果，不能直接套用到所有 ANN。

### 10.6 如何選擇 Learning Rate？

初學時可以採用以下流程：

1. 先選擇 Optimizer，例如 Adam。
2. 使用常見起始值，例如 `lr=0.001`。
3. 觀察 Training Loss 與 Validation Loss。
4. 如果 Loss 下降太慢，可以嘗試提高 Learning Rate。
5. 如果 Loss 大幅震盪或發散，可以嘗試降低 Learning Rate。
6. 使用 Validation Set 比較不同設定。

要注意，Loss 下降緩慢不一定是 Learning Rate 太小，也可能是資料前處理、模型容量或梯度等其他問題。

教材示意圖 06

- 圖片名稱：`06_Learning_Rate_Comparison.png`
- 圖片尺寸：1280 × 720 (16:9)
- 提示詞：製作 Learning Rate Too Small、Good Learning Rate、Learning Rate Too Large 三欄比較圖，三欄均使用 U 型 Loss 曲線與梯度下降路徑。第一欄小步緩慢前進，第二欄適當步長逐步接近最低點，第三欄步長過大反覆跨越最低點甚至發散，附繁體中文說明，白底、16:9。

## 十一、學習率排程器 (Learning Rate Scheduler)

### 11.1 什麼是 Learning Rate Scheduler？

前面的例子使用固定 Learning Rate：

```
optimizer = optim.Adam(    model.parameters(),    lr=0.001)
```

如果整個訓練過程都沒有修改 Learning Rate，模型就會一直使用相同的基礎學習率。

但在許多訓練情境中，我們希望：

```
訓練初期
    ↓
使用較大的 Learning Rate
    ↓
快速改善模型
    ↓
訓練後期
    ↓
逐漸降低 Learning Rate
    ↓
更細緻地調整參數
```

這時就可以使用 Learning Rate Scheduler。

Scheduler 的工作是依照預先設定的規則，或根據驗證指標，調整 Optimizer 的 Learning Rate。

### 11.2 StepLR

StepLR 會每隔指定的更新週期，將 Learning Rate 乘上一個比例。

例如：

```
from torch.optim.lr_scheduler import StepLR### 建立 Optimizeroptimizer = optim.Adam(    model.parameters(),    lr=0.01)### 建立 StepLRscheduler = StepLR(    optimizer,    step_size=20, # 每 20 次 scheduler.step() 調整一次    gamma=0.5 # Learning Rate 乘上 0.5)
```

如果每個 Epoch 呼叫一次 `scheduler.step()`：

| Epoch | Learning Rate |
| ----- | ------------- |
| 1～20  | 0.0100        |
| 21～40 | 0.0050        |
| 41～60 | 0.0025        |
| 61～80 | 0.00125       |

#### StepLR 參數

| 參數           | 說明                      |
| ------------ | ----------------------- |
| `optimizer`  | 要調整的 Optimizer          |
| `step_size`  | 每隔多少次 Scheduler Step 調整 |
| `gamma`      | 每次調整時的倍率                |
| `last_epoch` | 排程進度，通常維持預設 `-1`        |

### 11.3 ExponentialLR

ExponentialLR 每次呼叫 `scheduler.step()` 時，將 Learning Rate 乘上固定倍率。

```
from torch.optim.lr_scheduler import ExponentialLR### 建立 ExponentialLRscheduler = ExponentialLR(    optimizer,    gamma=0.95 # 每次調整後保留 95% 的 Learning Rate)
```

例如：

```
原本 LR = 0.0100
下一次 LR = 0.0095
再下一次 LR = 0.009025
```

這種方式會讓 Learning Rate 持續按比例下降。

### 11.4 CosineAnnealingLR

CosineAnnealingLR 使用餘弦曲線調整 Learning Rate。

在單一排程週期中，Learning Rate 通常會從初始值逐漸下降到指定的最小值附近。

```
from torch.optim.lr_scheduler import CosineAnnealingLR### 建立 CosineAnnealingLRscheduler = CosineAnnealingLR(    optimizer,    T_max=100, # 餘弦排程的週期長度，以 scheduler.step() 次數計    eta_min=0.00001 # 最小 Learning Rate)
```

#### CosineAnnealingLR 參數

| 參數           | 說明               |
| ------------ | ---------------- |
| `optimizer`  | 要調整的 Optimizer   |
| `T_max`      | 餘弦排程的半週期長度       |
| `eta_min`    | 最小 Learning Rate |
| `last_epoch` | 排程進度             |

需要注意，`CosineAnnealingLR` 不會自動執行 Warm Restarts。

如果希望定期重新提高 Learning Rate，可以另外研究 `CosineAnnealingWarmRestarts`。

### 11.5 ReduceLROnPlateau

前面的 Scheduler 主要根據預定的排程調整 Learning Rate。

但有時候我們希望根據 Validation Loss 的變化決定是否降低 Learning Rate。

這時可以使用：

```
from torch.optim.lr_scheduler import ReduceLROnPlateau### 建立 Schedulerscheduler = ReduceLROnPlateau(    optimizer,    mode="min", # 希望監控指標越小越好    factor=0.5, # 降低 Learning Rate 時乘上 0.5    patience=10, # 超過容忍的連續未改善次數後降低    threshold=1e-4, # 判定改善的門檻    min_lr=1e-6 # Learning Rate 下限)
```

例如：

```
Epoch 1～10
Validation Loss 持續改善
        ↓
維持 Learning Rate

後續連續多個 Epoch
Validation Loss 沒有足夠改善
        ↓
降低 Learning Rate
```

#### ReduceLROnPlateau 參數

| 參數               | 說明                            |
| ---------------- | ----------------------------- |
| `mode`           | `"min"` 代表越小越好，`"max"` 代表越大越好 |
| `factor`         | 降低 Learning Rate 的倍率          |
| `patience`       | 可容忍的連續未改善次數                   |
| `threshold`      | 判定指標有足夠改善的門檻                  |
| `threshold_mode` | 使用相對或絕對改善門檻                   |
| `cooldown`       | 降低 Learning Rate 後的冷卻期        |
| `min_lr`         | 最小 Learning Rate              |
| `eps`            | 避免過小的學習率調整                    |

### 11.6 Scheduler 應該放在哪裡？

一般 Epoch-based Scheduler：

```
for epoch in range(epochs):    ### 訓練模型    train_one_epoch()    ### 完成一個 Epoch 後更新 Learning Rate    scheduler.step()
```

ReduceLROnPlateau：

```
for epoch in range(epochs):    ### 訓練模型    train_one_epoch()    ### 計算 Validation Loss    valid_loss = validate_model()    ### 根據 Validation Loss 調整 Learning Rate    scheduler.step(valid_loss)
```

要注意，並不是所有 Scheduler 都應該每個 Epoch 呼叫一次。

有些 Scheduler 設計成每個 Batch 更新，例如部分 OneCycleLR 設定。

呼叫頻率必須配合 Scheduler 的設計與參數設定。

### 11.7 Scheduler 比較

| Scheduler         | 調整方式      | 適用情境       |
| ----------------- | --------- | ---------- |
| StepLR            | 每隔固定週期降低  | 希望簡單控制學習率  |
| ExponentialLR     | 每次按固定倍率降低 | 希望平滑的指數下降  |
| CosineAnnealingLR | 使用餘弦排程    | 已知大致訓練長度   |
| ReduceLROnPlateau | 監控指標停滯時降低 | 希望根據驗證表現調整 |

教材示意圖 07

- 圖片名稱：`07_Learning_Rate_Schedulers.png`
- 圖片尺寸：1280 × 720 (16:9)
- 提示詞：製作四種 Learning Rate Scheduler 曲線比較圖，四個子圖分別為 Constant LR、StepLR、ExponentialLR、CosineAnnealingLR，X 軸為 Epoch，Y 軸為 Learning Rate，使用清晰曲線呈現固定、階梯下降、指數下降與餘弦下降，標示每種 Scheduler 名稱，繁體中文教學說明，白底、16:9。

## 十二、PyTorch 完整實作：比較五種 Optimizer 的 ANN 房租預測效果

前面已經了解各種 Optimizer 的原理，接下來使用相同的 ANN 架構，比較：

- SGD
- SGD + Momentum
- RMSprop
- Adam
- AdamW

這次會使用 Mini-batch Gradient Descent，讓學生同時練習 `Dataset`、`DataLoader`、`optimizer.step()` 與 `scheduler.step()`。

### 12.1 準備資料集

optimizer_rent.csv

2,400 筆模擬房屋租金資料 · 5 個 Features · 1 個 Target

下載 CSV 資料集

這份資料是本章新建立的合成資料集，與前一章的房租資料不是同一份 CSV。

資料欄位：

| 欄位                   | 說明            |
| -------------------- | ------------- |
| `Area_Ping`          | 房屋坪數          |
| `Building_Age`       | 屋齡            |
| `Distance_To_MRT_Km` | 距離捷運站的公里數     |
| `Floor`              | 所在樓層          |
| `Rooms`              | 房間數           |
| `Monthly_Rent`       | 每月租金 (Target) |

資料包含非線性關係與隨機雜訊，適合比較不同 Optimizer 的訓練情況，但不能代表真實租屋市場價格。

### 12.2 實作流程

### 12.3 安裝與匯入套件

在 Windows + VSCode + uv 環境，可以先安裝：

如果使用 Jupyter Notebook，還需要確認目前 Notebook Kernel 指向正確的 Python 環境。

這裡固定隨機種子是為了讓實驗比較容易重現。

但 GPU 運算與部分底層演算法仍可能存在非決定性，因此不保證每次執行結果完全一致。

### 12.4 讀取資料集

### 12.5 資料清理

這份資料已經整理完成，這裡主要示範基本清理流程。

### 12.6 選擇 Features 與 Target

### 12.7 切分 Train / Validation / Test

本次使用 60% / 20% / 20%。

因為這是迴歸問題，Target 是連續數值，因此這裡沒有直接使用 `stratify=y`。

### 12.8 StandardScaler 標準化

這次同時標準化 Features 與 Target。

原因是原始租金可能有數萬元，直接使用 MSELoss 訓練時，Loss 與梯度尺度可能很大。

將 Target 標準化，有助於讓不同 Optimizer 的比較更容易進行。

這裡有兩個重點：

- Scaler 只能使用 Train Set 執行 `fit()`。
- 模型預測完成後，必須將租金還原成原始單位再評估。

### 12.9 轉換成 PyTorch Tensor

這裡刻意將訓練資料保留在 CPU，交給 DataLoader 分批載入。

每次訓練時，再把當前 Batch 搬到 GPU 或 CPU 運算裝置。

### 12.10 建立 TensorDataset 與 DataLoader

因為本次 Train Set 有 1,440 筆資料，因此：

\\[ \left\lceil\frac{1440}{64}\right\rceil=23 \\]

每個 Epoch 會執行 23 次參數更新。

### 12.11 建立 ANN Regression

這次使用：